# 07 -- Mediation Analysis (excluded from v2)

This notebook is intentionally **not** a working analysis in the v2 pipeline. It
documents what the original exploratory mediation analysis (`exploratory/07_mediation.ipynb`)
attempted and why it was dropped here, mirroring the same decision already made
for `CRC_Gut_Analysis_Git/05_mediation_network.ipynb` earlier in this project.

## What it tried

Species -> polyamine -> CRC-stage mediation, using `statsmodels.stats.mediation.Mediation`
(Imai-Keele-Tingley potential-outcomes simulation) for each (species, polyamine) pair drawn
from the top-5 SHAP-ranked "producer" species per polyamine.

## Why it was dropped, not fixed

**1. Structural circularity.** The mediator-selection step (top-5 SHAP producer species per
polyamine) and the mediation test itself (does that species -> polyamine path mediate the
stage outcome) both drew on the *same model, fit on the same cohort*. The species most likely
to pass the mediation test were selected, by construction, because they already looked like
strong producers in that exact dataset -- the a-path is close to guaranteed to look strong
regardless of the true causal structure. This is not a bug to patch; it is the same "circular
stream" reasoning that led to dropping mediation from the git-tracked pipeline's own deliverables.

**2. A real implementation bug compounded it.** `Mediation.fit()` hardcodes a treatment=0 vs
treatment=1 contrast (confirmed directly from the installed `statsmodels` source -- this is the
Imai-Keele-Tingley design, not classical Baron & Kenny, despite the original notebook's markdown
claiming the latter). The exposure (raw species abundance) was never rescaled into that 0-1
range, unlike the z-scored covariates in the same model -- so even setting circularity aside, the
reported ACME values were not real per-unit effects and were not comparable across species,
despite being ranked against each other in the forest plot.

**3. Its outcome variable inherited the disease-stage mislabeling bug.** The original notebook's
outcome was the old, incorrectly-ordered `CRC.Stage.Numeric` (MP -- multiple polypoid adenoma --
patients coded as *more* severe than Stage III/IV cancer) -- the single highest-stakes place that
bug reached in the whole pipeline, since this was a formal causal-effect estimate rather than a
descriptive correlation. `utils.py` fixes that encoding project-wide in v2 (and excludes HS, history of colorectal
surgery), but it does not fix
findings 1-2 above, which are why mediation is excluded here rather than repaired.

## What v2 does instead

Species-metabolite production evidence lives in `03_ml_benchmarking.ipynb` (SHAP, now using
bias-corrected out-of-fold values as the primary ranking) and `08_pathway_analysis.ipynb`
(AGORA2 species-encodes-enzyme evidence, replacing genus-level KEGG lookups). Neither claims a
causal/mediated pathway -- both are framed as predictive-association and genomic-capability
evidence, which is what the underlying data can actually support.
